"""
FIFA World Cup 2026 — Step 2: Feature Engineering
==================================================
Transforms raw match data into ML-ready features.
 
For every match, we compute:
  1. ELO difference        → team skill gap at match time
  2. Recent form           → win/draw rate over last 10 matches
  3. Goals average         → avg scored & conceded over last 10 matches
  4. Head-to-head record   → historical win rate between these two teams
  5. Tournament weight     → how important the match type is
  6. Neutral venue flag    → removes home advantage bias
 
Output files (saved to /data):
  - features_train.csv    → matches 2000–2021  (train the model)
  - features_test.csv     → 2022 World Cup      (evaluate accuracy)
  - features_2026.csv     → 2026 group fixtures (predict!)
"""

In [45]:
import pandas as pd
import numpy as np
import requests
import os
import itertools
import warnings
warnings.filterwarnings("ignore")


In [46]:
from pandas import read_csv
from datetime import timedelta


DATA_DIR = os.path.join(os.getcwd(), "data")

#load from data
match_results = pd.read_csv(os.path.join(DATA_DIR, "match_results.csv"), parse_dates=["date"])
elo_ratings = pd.read_csv(os.path.join(DATA_DIR, "elo_ratings.csv"), parse_dates=["date"])

print(f"Loaded {len(match_results)}matches")
print(f"Loaded {len(elo_ratings)} ELO records")
print("date range : {match_results['date'].min().date()} to {match_results['date'].max().date()} ")

Loaded 25268matches
Loaded 50536 ELO records
date range : {match_results['date'].min().date()} to {match_results['date'].max().date()} 


##  ELO Difference at Match Time

**Why:** ELO is the single strongest predictor. A 100-point advantage = ~70% win probability.

**How:** For each match, find the ELO rating of both teams at that exact date (or closest prior date)

In [47]:
def get_elo_at_date(team, match_date, elo_df):
    team_elo = elo_df[(elo_df["team"] == team) & (elo_df["date"] <= match_date)]
    
    if len(team_elo) == 0:
        return 1500  # Default ELO for new teams
    
    # Get the most recent ELO before the match
    return team_elo.sort_values("date").iloc[-1]["elo"]

# Test on a few matches
test_match = match_results.iloc[1000]
home_elo = get_elo_at_date(test_match["home_team"], test_match["date"], elo_ratings)
away_elo = get_elo_at_date(test_match["away_team"], test_match["date"], elo_ratings)

print(f"Sample match: {test_match['home_team']} vs {test_match['away_team']} ({test_match['date'].date()})")
print(f"  Home ELO: {home_elo:.0f}")
print(f"  Away ELO: {away_elo:.0f}")
print(f"  ELO diff: {home_elo - away_elo:.0f}")

Sample match: San Marino vs Latvia (2000-11-15)
  Home ELO: 1439
  Away ELO: 1445
  ELO diff: -5


---
##  Recent Form (Last 10 Matches)

**Why:** Captures momentum. A team on a 3-match winning streak is hotter than a team with the same historical record.

**How:** For each team & match date, compute win% from their last 10 matches *before* that date.

In [48]:
def get_recent_form(team, match_date, results_df, lookback=10):
    """
    Compute team's win rate in their last `lookback` matches before match_date.
    Returns: win_rate (0.0 to 1.0)
    """
    # Get all matches for this team before the match date
    team_matches = results_df[
        ((results_df["home_team"] == team) | (results_df["away_team"] == team)) &
        (results_df["date"] < match_date)
    ].sort_values("date")
    
    if len(team_matches) == 0:
        return 0.5  # Default: 50% for new teams
    
    # Take last `lookback` matches
    recent = team_matches.tail(lookback)
    
    # Calculate result from this team's perspective
    wins = 0
    for _, match in recent.iterrows():
        if match["home_team"] == team:
            if match["result"] == "W":
                wins += 1
            elif match["result"] == "D":
                wins += 0.5  # Draw is half a win
        else:  # Away team
            away_result = "L" if match["result"] == "W" else ("D" if match["result"] == "D" else "W")
            if away_result == "W":
                wins += 1
            elif away_result == "D":
                wins += 0.5
    
    return wins / len(recent)

# Test
home_form = get_recent_form(test_match["home_team"], test_match["date"], match_results)
away_form = get_recent_form(test_match["away_team"], test_match["date"], match_results)

print(f"Recent form (last 10 matches before {test_match['date'].date()}):")
print(f"  {test_match['home_team']}: {home_form:.2%}")
print(f"  {test_match['away_team']}: {away_form:.2%}")

Recent form (last 10 matches before 2000-11-15):
  San Marino: 0.00%
  Latvia: 14.29%


---
## Goals Scored & Conceded Average

**Why:** Captures attacking and defensive strength better than ELO alone.

**How:** For each team, average goals scored/conceded in last 10 matches before the game.

In [49]:
def get_goals_average(team, match_date, results_df, lookback=10):
    """
    Compute team's average goals scored and conceded in last `lookback` matches.
    Returns: (avg_scored, avg_conceded)
    """
    team_matches = results_df[
        ((results_df["home_team"] == team) | (results_df["away_team"] == team)) &
        (results_df["date"] < match_date)
    ].sort_values("date")
    
    if len(team_matches) == 0:
        return 1.0, 1.0  # Default: 1 goal per side
    
    recent = team_matches.tail(lookback)
    
    scored = 0
    conceded = 0
    
    for _, match in recent.iterrows():
        if match["home_team"] == team:
            scored += match["home_score"]
            conceded += match["away_score"]
        else:
            scored += match["away_score"]
            conceded += match["home_score"]
    
    return scored / len(recent), conceded / len(recent)

# Test
home_goals_for, home_goals_against = get_goals_average(test_match["home_team"], test_match["date"], match_results)
away_goals_for, away_goals_against = get_goals_average(test_match["away_team"], test_match["date"], match_results)

print(f"Goals average (last 10 matches before {test_match['date'].date()}):")
print(f"  {test_match['home_team']}: {home_goals_for:.2f} scored, {home_goals_against:.2f} conceded")
print(f"  {test_match['away_team']}: {away_goals_for:.2f} scored, {away_goals_against:.2f} conceded")

Goals average (last 10 matches before 2000-11-15):
  San Marino: 0.00 scored, 1.50 conceded
  Latvia: 0.43 scored, 1.71 conceded


---
## Head-to-Head Record

**Why:** Some rivalries favor one team. Direct history can predict future matches.

**How:** For each pair of teams, compute home team's win rate in all *previous* H2H matchups.

In [50]:
def get_h2h_record(home_team, away_team, match_date, results_df):
    """
    Compute home team's win rate against away_team in all prior H2H matches.
    Returns: home team's win rate (0.0 to 1.0)
    """
    h2h = results_df[
        (
            ((results_df["home_team"] == home_team) & (results_df["away_team"] == away_team)) |
            ((results_df["home_team"] == away_team) & (results_df["away_team"] == home_team))
        ) &
        (results_df["date"] < match_date)
    ]
    
    if len(h2h) == 0:
        return 0.5  # Default: even record if no prior match
    
    home_wins = 0
    for _, match in h2h.iterrows():
        # Check if home_team was actually at home
        if match["home_team"] == home_team:
            if match["result"] == "W":
                home_wins += 1
            elif match["result"] == "D":
                home_wins += 0.5
        else:  # home_team was away
            away_result = "L" if match["result"] == "W" else ("D" if match["result"] == "D" else "W")
            if away_result == "W":
                home_wins += 1
            elif away_result == "D":
                home_wins += 0.5
    
    return home_wins / len(h2h)

# Test
h2h = get_h2h_record(test_match["home_team"], test_match["away_team"], test_match["date"], match_results)
print(f"H2H record: {test_match['home_team']} vs {test_match['away_team']}")
print(f"  {test_match['home_team']} win rate at home: {h2h:.2%}")

H2H record: San Marino vs Latvia
  San Marino win rate at home: 50.00%


---
##  Tournament Weight

**Why:** World Cup matches matter more than friendlies. Weights:
- World Cup: 1.0
- Qualifier: 0.7
- Friendly: 0.3

**How:** Check tournament type in the data and assign weight.

In [51]:
def get_tournament_weight(tournament_name):
    """
    Assign importance weight to a match based on tournament type.
    """
    if "World Cup" in str(tournament_name):
        return 1.0
    elif "Qualifying" in str(tournament_name) or "Qualification" in str(tournament_name):
        return 0.7
    elif "Championship" in str(tournament_name) or "Copa" in str(tournament_name):
        return 0.9  # Continental tournaments are important
    else:
        return 0.3  # Friendlies are less important

# Test
print(f"Tournament weights:")
sample_tournaments = match_results["tournament"].unique()[:5]
for t in sample_tournaments:
    weight = get_tournament_weight(t)
    print(f"  {t}: {weight}")

Tournament weights:
  Friendly: 0.3
  African Cup of Nations: 0.3
  AFC Asian Cup qualification: 0.3
  Nordic Championship: 0.9
  Cyprus International Tournament: 0.3


---
##  Assemble Feature Matrix

Now combine all features into one DataFrame, one row per match.

⚠️ **Warning:** This is computationally intensive. For each match, we scan ELO history for both teams (can be slow).
Consider this a one-time cost.

In [52]:
def engineer_features(matches_df, elo_df):
    features=[]
    total =len(matches_df)
    for idx, row in matches_df.iterrows():
        if (idx+1) % 500 == 0:
            print(f" Processing match {idx +1}/{total}")
        match_date = row["date"]
        home_team = row["home_team"]
        away_team = row["away_team"]

        # ELO difference
        home_elo = get_elo_at_date(home_team, match_date, elo_df)
        away_elo = get_elo_at_date(away_team, match_date, elo_df)
        elo_diff= home_elo - away_elo

        #Recent form
        home_form = get_recent_form(home_team, match_date, matches_df)
        away_form = get_recent_form(away_team, match_date, matches_df)
        
        # Goals average
        home_goals_for, home_goals_against = get_goals_average(home_team, match_date, matches_df)
        away_goals_for, away_goals_against = get_goals_average(away_team, match_date, matches_df)

        # H2H record
        h2h_home_win_rate = get_h2h_record(home_team, away_team, match_date, matches_df)

        # Tournament weight
        tournament_weight = get_tournament_weight(row["tournament"])

        # neutral venue
        is_neutral = int(row['neutral'])

        #target result
        result = row["result"]

        # Elo win probability
        elo_win_prob = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))

        # Form features
        home_form_win = home_form
        away_form_win = away_form

        # Approximate draw tendency
        home_form_draw = 1 - home_form
        away_form_draw = 1 - away_form

        form_diff = home_form - away_form

        # Goal features
        home_avg_scored = home_goals_for
        home_avg_conceded = home_goals_against

        away_avg_scored = away_goals_for
        away_avg_conceded = away_goals_against

        # Expected goal difference
        xgd = (
            (home_avg_scored - home_avg_conceded)
            - (away_avg_scored - away_avg_conceded)
        )

        # H2H draw rate
        h2h_matches = matches_df[
            (
                ((matches_df["home_team"] == home_team) & (matches_df["away_team"] == away_team))
                |
                ((matches_df["home_team"] == away_team) & (matches_df["away_team"] == home_team))
            )
            &
            (matches_df["date"] < match_date)
        ]

        if len(h2h_matches) > 0:
            h2h_draw_rate = (h2h_matches["result"] == "D").mean()
        else:
            h2h_draw_rate = 0.33

        # World Cup flag
        is_worldcup = int("World Cup" in str(row["tournament"]))

        features.append({
            "date": match_date,
            "home_team": home_team,
            "away_team": away_team,
            "home_elo": home_elo,
            "away_elo": away_elo,
            "elo_diff": elo_diff,
            "home_form": home_form,
            "away_form": away_form,
            "home_goals_for": home_goals_for,
            "home_goals_against": home_goals_against,
            "away_goals_for": away_goals_for,
            "away_goals_against": away_goals_against,
            "h2h_home_win_rate": h2h_home_win_rate,
            "tournament_weight": tournament_weight,
            "is_neutral": is_neutral,
            "result": result,
            "elo_win_prob": elo_win_prob,
            "home_form_win": home_form_win,
            "home_form_draw": home_form_draw,
             "away_form_win": away_form_win,
            "away_form_draw": away_form_draw,
            "form_diff": form_diff,

            "home_avg_scored": home_avg_scored,
            "home_avg_conceded": home_avg_conceded,
            "away_avg_scored": away_avg_scored,
            "away_avg_conceded": away_avg_conceded,

            "xgd": xgd,

            "h2h_home_win_rate": h2h_home_win_rate,
            "h2h_draw_rate": h2h_draw_rate,
            "tournament_weight": tournament_weight,
            "is_worldcup": is_worldcup,

        })

    return pd.DataFrame(features)
print("engineering features...")
features_df = engineer_features(match_results, elo_ratings)

print(f"\n features engineered for {len(features_df)} matches")
print(f" Feature matrix shape: { features_df.shape}")
print(f"\n first 5 rows: ")
print(features_df.head())




engineering features...
 Processing match 500/25268
 Processing match 1000/25268
 Processing match 1500/25268
 Processing match 2000/25268
 Processing match 2500/25268
 Processing match 3000/25268
 Processing match 3500/25268
 Processing match 4000/25268
 Processing match 4500/25268
 Processing match 5000/25268
 Processing match 5500/25268
 Processing match 6000/25268
 Processing match 6500/25268
 Processing match 7000/25268
 Processing match 7500/25268
 Processing match 8000/25268
 Processing match 8500/25268
 Processing match 9000/25268
 Processing match 9500/25268
 Processing match 10000/25268
 Processing match 10500/25268
 Processing match 11000/25268
 Processing match 11500/25268
 Processing match 12000/25268
 Processing match 12500/25268
 Processing match 13000/25268
 Processing match 13500/25268
 Processing match 14000/25268
 Processing match 14500/25268
 Processing match 15000/25268
 Processing match 15500/25268
 Processing match 16000/25268
 Processing match 16500/25268
 Proce

In [53]:
teams_2026 = pd.read_csv(os.path.join(DATA_DIR, "2026_teams.csv"))
# Generate 2026 group stage fixtures
fixtures = []

for group, group_df in teams_2026.groupby("group"):
    teams = group_df["team"].tolist()

    # Every pair of teams plays once
    for home_team, away_team in itertools.combinations(teams, 2):
        fixtures.append({
            "group": group,
            "home_team": home_team,
            "away_team": away_team
        })

fixtures_2026 = pd.DataFrame(fixtures)

print(f"Generated {len(fixtures_2026)} group-stage fixtures")

Generated 72 group-stage fixtures


In [58]:
rows = []

reference_date = match_results["date"].max()

for _, row in fixtures_2026.iterrows():

    home_team = row["home_team"]
    away_team = row["away_team"]
    

    home_elo = get_elo_at_date(home_team, reference_date, elo_ratings)
    away_elo = get_elo_at_date(away_team, reference_date, elo_ratings)

    home_form = get_recent_form(home_team, reference_date, match_results)
    away_form = get_recent_form(away_team, reference_date, match_results)

    home_goals_for, home_goals_against = get_goals_average(
        home_team, reference_date, match_results
    )

    away_goals_for, away_goals_against = get_goals_average(
        away_team, reference_date, match_results
    )

    h2h = get_h2h_record(
        home_team,
        away_team,
        reference_date,
        match_results
    )
    elo_win_prob = 1 / (
        1 + 10 ** ((away_elo - home_elo) / 400)
        )

    form_diff = home_form - away_form

    home_avg_scored = home_goals_for
    home_avg_conceded = home_goals_against

    away_avg_scored = away_goals_for
    away_avg_conceded = away_goals_against

    xgd = (
        (home_avg_scored - home_avg_conceded)
        -
        (away_avg_scored - away_avg_conceded)
    )

    


    rows.append({
    "group": group,
    "home_team": home_team,
    "away_team": away_team,

    "home_elo": home_elo,
    "away_elo": away_elo,
    "elo_diff": home_elo - away_elo,
    "elo_win_prob": elo_win_prob,

    "home_form_win": home_form,
    "home_form_draw": 1 - home_form,

    "away_form_win": away_form,
    "away_form_draw": 1 - away_form,

    "form_diff": form_diff,

    "home_avg_scored": home_avg_scored,
    "home_avg_conceded": home_avg_conceded,

    "away_avg_scored": away_avg_scored,
    "away_avg_conceded": away_avg_conceded,

    "xgd": xgd,

    "h2h_home_win_rate": h2h,

    # No historical H2H draw feature available
    "h2h_draw_rate": 0.33,

    "tournament_weight": 1.0,
    "is_neutral": 1,
    "is_worldcup": 1
    })

features_2026 = pd.DataFrame(rows)

---
##  Split into Train & Test Sets

- **Train:** 2000–2021 matches (to train the model)
- **Test:** 2022 World Cup matches (to evaluate accuracy)

This temporal split prevents data leakage (model never sees 2022 data during training).

In [59]:
# Split by year
features_df["year"] = features_df["date"].dt.year

# 2022 World Cup matches (test set)
test_set = features_df[features_df["year"] == 2022].copy()

# 2000-2021 matches (training set)
train_set = features_df[features_df["year"] < 2022].copy()



# Remove the temporary year column
train_set = train_set.drop("year", axis=1)
test_set = test_set.drop("year", axis=1)

print(f"Training set: {len(train_set):,} matches (2000–2021)")
print(f"Test set: {len(test_set):,} matches (2022)")

print(f"\nTest set matches (2022 World Cup):")
print(test_set[["date", "home_team", "away_team", "result"]].head(10))

Training set: 20,775 matches (2000–2021)
Test set: 969 matches (2022)

Test set matches (2022 World Cup):
            date   home_team     away_team result
20775 2022-01-01    Thailand     Indonesia      D
20776 2022-01-02       Gabon  Burkina Faso      L
20777 2022-01-02       Sudan      Zimbabwe      D
20778 2022-01-03      Rwanda        Guinea      W
20779 2022-01-04  Mauritania         Gabon      D
20780 2022-01-05     Algeria         Ghana      W
20781 2022-01-06      Rwanda        Guinea      L
20782 2022-01-09    Cameroon  Burkina Faso      W
20783 2022-01-09    Ethiopia    Cape Verde      L
20784 2022-01-10     Senegal      Zimbabwe      W


In [60]:
# Save training features
train_path = os.path.join(DATA_DIR, "features_train.csv")
train_set.to_csv(train_path, index=False)
print(f"✓ Training features saved → {train_path}")

# Save test features
test_path = os.path.join(DATA_DIR, "features_test.csv")
test_set.to_csv(test_path, index=False)
print(f"✓ Test features saved → {test_path}")


features_2026_path = os.path.join(DATA_DIR, "features_2026.csv")
features_2026.to_csv(features_2026_path,index=False)
print(f"✓ 2026 features saved → {features_2026_path}")

# Show statistics
print(f"\n" + "="*50)
print("FEATURE STATISTICS")
print("="*50)
print(f"\nTraining set shape: {train_set.shape}")
print(f"Test set shape: {test_set.shape}")

print(f"\nFeature ranges (training set):")
numeric_cols = ["elo_diff", "home_form", "away_form", "home_goals_for", 
                "away_goals_for", "h2h_home_win_rate", "tournament_weight"]
print(train_set[numeric_cols].describe().round(3))


✓ Training features saved → c:\Users\RIVU\Projects\FIFA Prediction\data\features_train.csv
✓ Test features saved → c:\Users\RIVU\Projects\FIFA Prediction\data\features_test.csv
✓ 2026 features saved → c:\Users\RIVU\Projects\FIFA Prediction\data\features_2026.csv

FEATURE STATISTICS

Training set shape: (20775, 29)
Test set shape: (969, 29)

Feature ranges (training set):
        elo_diff  home_form  away_form  home_goals_for  away_goals_for  \
count  20775.000  20775.000  20775.000       20775.000       20775.000   
mean      17.572      0.507      0.493           1.401           1.360   
std      167.749      0.197      0.198           0.709           0.711   
min     -979.147      0.000      0.000           0.000           0.000   
25%      -79.431      0.400      0.350           1.000           0.900   
50%       19.842      0.500      0.500           1.300           1.300   
75%      118.764      0.650      0.650           1.700           1.700   
max      973.102      1.000      1

In [61]:
# Check for missing values
print("Missing values in training set:")
missing = train_set.isnull().sum()
if missing.sum() == 0:
    print("  ✓ No missing values")
else:
    print(missing[missing > 0])

print("\nMissing values in test set:")
missing_test = test_set.isnull().sum()
if missing_test.sum() == 0:
    print("  ✓ No missing values")
else:
    print(missing_test[missing_test > 0])

# Check result distribution
print(f"\nResult distribution (training set):")
print(train_set["result"].value_counts())
print(f"\nResult distribution (test set):")
print(test_set["result"].value_counts())

# Show sample features
print(f"\n" + "="*50)
print("SAMPLE MATCH WITH ALL FEATURES")
print("="*50)
sample = train_set.iloc[1000]
print(f"\nMatch: {sample['home_team']} vs {sample['away_team']} ({sample['date'].date()})")
print(f"Result: {sample['result']}")
print(f"\nFeatures:")
print(f"  ELO difference: {sample['elo_diff']:.0f} (home advantage)")
print(f"  Home form: {sample['home_form']:.1%} | Away form: {sample['away_form']:.1%}")
print(f"  Home goals (for/against): {sample['home_goals_for']:.2f} / {sample['home_goals_against']:.2f}")
print(f"  Away goals (for/against): {sample['away_goals_for']:.2f} / {sample['away_goals_against']:.2f}")
print(f"  H2H home win rate: {sample['h2h_home_win_rate']:.1%}")
print(f"  Tournament weight: {sample['tournament_weight']}")
print(f"  Neutral venue: {bool(sample['is_neutral'])}")

Missing values in training set:
  ✓ No missing values

Missing values in test set:
  ✓ No missing values

Result distribution (training set):
result
W    10019
L     5900
D     4856
Name: count, dtype: int64

Result distribution (test set):
result
W    482
L    267
D    220
Name: count, dtype: int64

SAMPLE MATCH WITH ALL FEATURES

Match: San Marino vs Latvia (2000-11-15)
Result: L

Features:
  ELO difference: -5 (home advantage)
  Home form: 0.0% | Away form: 14.3%
  Home goals (for/against): 0.00 / 1.50
  Away goals (for/against): 0.43 / 1.71
  H2H home win rate: 50.0%
  Tournament weight: 1.0
  Neutral venue: False
